# Deep Fashion Dataset Processing

In [ ]:
import pandas as pd

In [ ]:
clothes_attr_path = '/home/kwesi/Desktop/Gazi/multi_label/data/DeepFashion/Category and Attribute Prediction Benchmark/Anno_coarse/list_attr_cloth.txt'

In [ ]:
df_clothes_attr = pd.read_csv(clothes_attr_path, skiprows=2, sep=r'\s{2,}', names=['attribute_name', 'attribute_type'], engine='python')

In [ ]:
images_attr_path = '/home/kwesi/Desktop/Gazi/multi_label/data/DeepFashion/Category and Attribute Prediction Benchmark/Anno_coarse/list_attr_img.txt'

In [ ]:
with open(images_attr_path, 'r') as file:
    num_rows = int(file.readline().strip())
    column_names = str(file.readline().strip())
    df_images_attr = pd.read_csv(file, sep=r'\s{3,}', names=['image_name', 'attribute_labels'], engine='python')

df_images_attr.iloc[:, :] = df_images_attr.iloc[:, :].replace(['', None], -1).astype(str)
attribute_columns = df_images_attr['attribute_labels'].str.split(' ', expand=True)
attribute_columns = attribute_columns.rename(columns=lambda x: f'attribute_{x}')
df_images_attr = pd.concat([df_images_attr[['image_name']], attribute_columns], axis=1)

In [ ]:
unique_values_all = {col: df_images_attr[col].unique() for col in df_images_attr.columns}

In [ ]:
df_images_attr = df_images_attr.iloc[:, :1001]

In [ ]:
df_images_attr_numeric = df_images_attr.iloc[:, 1:]

In [ ]:
numeric_df = df_images_attr.iloc[:,1:]
count_ones = (numeric_df == '1').sum()
top_40_columns = count_ones.sort_values(ascending=False).head(40)
top_40_columns

In [ ]:
df_images_attr_numeric.iloc[:, :] = df_images_attr_numeric.iloc[:, :].replace(['', None], -1).astype(int)
df_images_attr.iloc[:, 1:] = df_images_attr_numeric

In [ ]:
numeric_df = df_images_attr.iloc[:, 1:]
count_ones = (numeric_df == 1).sum()
top_40_columns = count_ones.sort_values(ascending=False).head(40)

In [ ]:
extracted_numbers = top_40_columns.index.str.extract(r'attribute_(\d+)').astype(int)
result = pd.DataFrame({
    'attribute': extracted_numbers[0]
})
result = result.sort_values(by='attribute').reset_index(drop=True)


In [ ]:
index_list = result.values.tolist()
index_list = [x[0]+1 for x in index_list]
index_list.insert(0, 0)

In [ ]:
df_images_attr_top_40 = df_images_attr.iloc[:, index_list]

In [ ]:
df_images_attr_top_40.columns = [df_images_attr_top_40.columns[0]] + [
    df_clothes_attr.iat[int(x.split('_')[1]), 0] for x in df_images_attr_top_40.columns[1:]
]

In [ ]:
df_images_attr_top_40.iloc[:, 0] = df_images_attr_top_40.iloc[:, 0].str.replace(r'[",\n]', '', regex=True)

In [ ]:
df_images_attr_top_40[df_images_attr_top_40['image_name'].str.contains(',', na=False)]

In [ ]:
df_images_attr_top_40.to_csv('/home/kwesi/Desktop/Gazi/multi_label/data/DeepFashion/Category and Attribute Prediction Benchmark/Anno_coarse/list_attr_img_top_40.txt', sep=',', index=False)

In [ ]:
images_partition_path = '/home/kwesi/Desktop/Gazi/multi_label/data/DeepFashion/Category and Attribute Prediction Benchmark/list_eval_partition.txt'

In [ ]:
with open(images_partition_path, 'r') as file:
    num_rows = int(file.readline().strip())
    column_names = str(file.readline().strip())
    df_images_partition = pd.read_csv(file, sep=r'\s{2,}', names=['image_name', 'evaluation_status'], engine='python')
    df_images_partition.iloc[:, 0] = df_images_partition.iloc[:, 0].str.replace(r'[",\n]', '', regex=True)
    df_images_partition.to_csv('/home/kwesi/Desktop/Gazi/multi_label/data/DeepFashion/Category and Attribute Prediction Benchmark/Anno_coarse/list_partition_top_40.txt', sep=',', index=False)